# Qwen3-8B-base-LoRA — two-stage (public pool → host)

Same **LoRA24** recipe as [`Qwen3-8B-base-Lora.ipynb`](Qwen3-8B-base-Lora.ipynb), trained in two stages, following the public notebook that went ~0.48 → 0.569 local CV with this exact pool:

1. **Stage 1 — read a lot:** 1 epoch on **`pool_v1.parquet`** (34,940 public passages, 18 models, 14 sources), class-weighted CE because the pool is unbalanced (1,155–2,713 per model).
2. **Stage 2 — listen to the host:** continue from the stage‑1 adapter; **2 epochs** on competition `train.csv` at lower LR; 5 prompt folds + `final`. No pool passages mixed back in.

Previous runs: host-only **0.489**, two-stage with the small extended set **0.487**. The pool already contains that extended set (source `prvsiyan`) plus ~30.7k other passages. It shares no text or prompts with `train.csv`.

## Runtime (2× T4)
| Phase | Work | Estimate |
| --- | --- | --- |
| Stage 1 | 1 epoch on ~34.9k passages, **1 GPU** | ~4–4.5 h |
| Stage 2 | 2 epochs on host, folds `0–4` + `final`, 2 GPUs | ~2–2.5 h |
| **Total** | | **~6.5–7 h** |

To save ~1.5 h, set `S2_JOBS = {0: ['final'], 1: ['0']}` (final adapter + one validation fold).

## Attach on Kaggle
- Competition **Name That LLM**
- **Qwen3-8B-Base**
- Dataset **`nishantdahal/ntl-two-stage-lora`** (contains `pool_v1.parquet`)
- **2× T4**, internet off

## Hyperparameters


In [ ]:
SEED = 7
FOLD_SEED = 0

# Stage 1: public pool pretrain
S1_EPOCHS = 1
S1_LR_ADAPTER = 1e-4
S1_LR_HEAD = 1e-3
S1_USE_CLASS_WEIGHTS = True

# Stage 2: host fine-tune (lower LR, fewer epochs than host-only 3-epoch run)
S2_EPOCHS = 2
S2_LR_ADAPTER = 5e-5
S2_LR_HEAD = 5e-4
S2_JOBS = {
    0: ['0', '2', '4'],       # GPU 0
    1: ['1', '3', 'final'],  # GPU 1
}

# Shared recipe (match host LoRA24)
LAYERS = [2, 4, 6, 8, 12, 16, 20, 24]
KEEP = 24
MAX_LEN = 447
ACCUM = 6
WD = 0.01
WARMUP = 0.10
R = 16
ALPHA = 32
LDROP = 0.05

import glob
import json
import os
import pathlib
import subprocess
import sys
import time

import numpy as np
import pandas as pd

## Prepare the pool CSV

Loads `pool_v1.parquet`, keeps prompts with answers from at least **3** models (triplets need 3), and writes `pool_train.csv` (`id, prompt_id, text, model`) for the stage-1 worker.

In [ ]:
POOL_PATH = pathlib.Path('/kaggle/input/datasets/nishantdahal/ntl-two-stage-lora/pool_v1.parquet')


def find_pool() -> pathlib.Path:
    if POOL_PATH.is_file():
        return POOL_PATH
    for root in (pathlib.Path('/kaggle/input'), pathlib.Path('data'), pathlib.Path('../data')):
        if root.is_dir():
            hits = sorted(root.rglob('pool_v1.parquet'))
            if hits:
                return hits[0]
    raise FileNotFoundError('pool_v1.parquet not found - attach nishantdahal/ntl-two-stage-lora')


pool_path = find_pool()
pool = pd.read_parquet(pool_path)
need = {'id', 'prompt_id', 'text', 'model'}
missing = need - set(pool.columns)
assert not missing, missing

n_per = pool.groupby('prompt_id')['model'].nunique()
ok = set(n_per[n_per >= 3].index)
stage1_df = pool[pool['prompt_id'].isin(ok)][['id', 'prompt_id', 'text', 'model']].dropna().copy()
stage1_df['text'] = stage1_df['text'].astype(str)
assert stage1_df.model.nunique() == 18, sorted(stage1_df.model.unique())
stage1_df.to_csv('pool_train.csv', index=False)

print('pool source', pool_path)
print('pool rows', len(pool), '-> stage1 rows', len(stage1_df), 'prompts', stage1_df.prompt_id.nunique())
if 'source' in pool.columns:
    print(pool.loc[stage1_df.index, 'source'].value_counts().to_string())
print(stage1_df.model.value_counts().sort_index().to_string())

## Worker script

Same dual-GPU idea as the host-only notebook. Extras:

- `init_adapter`: continue from stage‑1 LoRA (+ optional heads/norms)
- `class_weights`: inverse-frequency weights for unbalanced extended data
- `out_root`: where to write `job_*` / `stage1` folders


### Worker 1 — imports, data, folds, tokenize


In [ ]:
%%writefile worker.py
import importlib.util as _iu
import json
import math
import os
import random
import sys
import time

import numpy as np
import pandas as pd
import torch
from safetensors.torch import load_file, save_file
from torch import nn
from transformers import AutoConfig, AutoModel, AutoTokenizer

_fs = _iu.find_spec
_iu.find_spec = lambda name, *a, **k: None if name.split('.')[0] == 'torchao' else _fs(name, *a, **k)

from peft import LoraConfig, PeftModel, get_peft_model
import peft.import_utils as _pi

_pi.is_torchao_available = lambda *a, **k: False

cfg = json.loads(sys.argv[1])
jobs = sys.argv[2].split(',')

train = pd.read_csv(cfg['train_csv'])
LABELS = sorted(train.model.unique())
label_to_i = {l: i for i, l in enumerate(LABELS)}
y = train.model.map(label_to_i).values
pid = train.prompt_id.values

prompts = np.array(sorted(train.prompt_id.unique()))
rng0 = np.random.RandomState(cfg['fold_seed'])
rng0.shuffle(prompts)
fold = train.prompt_id.map({p: i % 5 for i, p in enumerate(prompts)}).values

tok = AutoTokenizer.from_pretrained(cfg['model_dir'])
tok.padding_side = 'right'
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

tokens = [
    ids + [tok.eos_token_id]
    for ids in tok(
        train.text.tolist(),
        add_special_tokens=False,
        truncation=True,
        max_length=cfg['max_len'],
    )['input_ids']
]

L = cfg['layers']
dev = torch.device('cuda:0')

# class weights (stage 1)
ce_weight = None
if cfg.get('class_weights'):
    counts = train.model.value_counts().reindex(LABELS).fillna(1).values.astype(np.float64)
    w = counts.sum() / (len(LABELS) * counts)
    ce_weight = torch.tensor(w, dtype=torch.float32, device=dev)
    print('class_weights', {l: float(x) for l, x in zip(LABELS, w)}, flush=True)


### Worker 2 — `triplets`


In [ ]:
%%writefile -a worker.py

def triplets(rows, rng):
    out = []
    for p in np.unique(pid[rows]):
        m = rng.permutation(rows[pid[rows] == p])
        if len(m) < 3:
            continue
        for s in range(0, len(m), 3):
            b = list(m[s:s + 3])
            while len(b) < 3:
                b.append(int(rng.choice([x for x in m if x not in b])))
            out.append(b)
    if not out:
        raise RuntimeError('no triplets — need >=3 answers per prompt in the fit set')
    out = [out[i] for i in rng.permutation(len(out))]
    while len(out) % cfg['accum']:
        out.append(out[rng.randint(len(out))])
    return out


### Worker 3 — job setup (optional `init_adapter` + head warm-start)


In [ ]:
%%writefile -a worker.py

out_root = cfg.get('out_root', 'lora_out')

for job in jobs:
    t0 = time.time()
    outdir = os.path.join(out_root, f'job_{job}')
    os.makedirs(outdir, exist_ok=True)

    random.seed(cfg['seed'])
    np.random.seed(cfg['seed'])
    torch.manual_seed(cfg['seed'])
    torch.cuda.manual_seed_all(cfg['seed'])
    rng = np.random.RandomState(cfg['seed'] + (99 if job in ('final', 'stage1') else int(job)))

    if job in ('final', 'stage1'):
        fit = np.arange(len(train))
        valid = np.arange(min(3, len(train)))  # dummy for stage1/final path
    else:
        fit = np.where(fold != int(job))[0]
        valid = np.where(fold == int(job))[0]

    mc = AutoConfig.from_pretrained(cfg['model_dir'])
    mc.num_hidden_layers, mc.use_cache = cfg['keep'], False
    if getattr(mc, 'layer_types', None) is not None:
        mc.layer_types = mc.layer_types[:cfg['keep']]

    base = AutoModel.from_pretrained(cfg['model_dir'], config=mc, dtype=torch.float16).to(dev)
    base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    base.enable_input_require_grads()

    lora_cfg = LoraConfig(
        r=cfg['r'],
        lora_alpha=cfg['alpha'],
        lora_dropout=cfg['ldrop'],
        bias='none',
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    )
    init_adapter = cfg.get('init_adapter')
    if init_adapter:
        model = PeftModel.from_pretrained(base, init_adapter, is_trainable=True)
        print(f'job {job}: loaded init_adapter {init_adapter}', flush=True)
    else:
        model = get_peft_model(base, lora_cfg)

    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()

    norms = nn.ModuleList([nn.LayerNorm(4096) for _ in L]).to(dev).float()
    plain = nn.Linear(4096 * len(L), len(LABELS)).to(dev).float()
    centered = nn.Linear(4096 * len(L), len(LABELS)).to(dev).float()
    drop = nn.Dropout(0.1)

    # warm-start heads/norms from stage1 when present
    if init_adapter:
        npath = os.path.join(init_adapter, '..', 'norms.safetensors')
        # init_adapter is .../job_stage1/adapter → norms at .../job_stage1/norms.safetensors
        stage_dir = os.path.dirname(init_adapter.rstrip('/\\'))
        nfile = os.path.join(stage_dir, 'norms.safetensors')
        rfile = os.path.join(stage_dir, 'readouts.npz')
        if os.path.isfile(nfile):
            nsd = load_file(nfile)
            norms.load_state_dict({k.split('norms.', 1)[-1]: v for k, v in nsd.items()})
            print(f'job {job}: loaded norms from {nfile}', flush=True)
        if os.path.isfile(rfile):
            with np.load(rfile, allow_pickle=False) as z:
                classes = [str(x) for x in z['classes'].tolist()]
                assert classes == LABELS, (classes, LABELS)
                with torch.no_grad():
                    plain.weight.copy_(torch.tensor(z['plain_coef']))
                    plain.bias.copy_(torch.tensor(z['plain_intercept']))
                    centered.weight.copy_(torch.tensor(z['center_coef']))
                    centered.bias.copy_(torch.tensor(z['center_intercept']))
            print(f'job {job}: loaded heads from {rfile}', flush=True)

    ad = [p for p in model.parameters() if p.requires_grad]
    hd = list(norms.parameters()) + list(plain.parameters()) + list(centered.parameters())
    opt = torch.optim.AdamW(
        [{'params': ad, 'lr': cfg['lr_adapter']}, {'params': hd, 'lr': cfg['lr_head']}],
        weight_decay=cfg['wd'],
    )
    plan = [triplets(fit, rng) for _ in range(cfg['epochs'])]
    total = sum(len(e) for e in plan) // cfg['accum']
    warm = max(1, int(cfg['warmup'] * total))
    sch = torch.optim.lr_scheduler.LambdaLR(
        opt,
        lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))),
    )
    scaler = torch.amp.GradScaler(init_scale=256.0)


### Worker 4 — train loop (optional class-weighted CE)


In [ ]:
%%writefile -a worker.py

    def rep(idx):
        batch = tok.pad({'input_ids': [tokens[i] for i in idx]}, return_tensors='pt').to(dev)
        with torch.autocast('cuda', dtype=torch.float16):
            hs = model(**batch, output_hidden_states=True, use_cache=False).hidden_states
        m = batch['attention_mask'].unsqueeze(-1).float()
        m[:, 0] = 0
        with torch.autocast('cuda', enabled=False):
            return torch.cat(
                [nm((hs[l].float() * m).sum(1) / m.sum(1)) for l, nm in zip(L, norms)],
                -1,
            )

    model.train()
    norms.train()
    step = 0
    skipped = 0
    losses = []
    for ep, batches in enumerate(plan):
        for i, b in enumerate(batches):
            z = rep(b)
            yy = torch.as_tensor(y[b], device=dev)
            loss = nn.functional.cross_entropy(plain(drop(z)), yy, weight=ce_weight) + nn.functional.cross_entropy(
                centered(drop(z - z.mean(0, keepdim=True))), yy, weight=ce_weight
            )
            scaler.scale(loss / cfg['accum']).backward()
            losses.append(float(loss.detach()))
            if (i + 1) % cfg['accum'] == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(ad + hd, 1.0)
                before = scaler.get_scale()
                scaler.step(opt)
                scaler.update()
                if scaler.get_scale() < before:
                    skipped += 1
                opt.zero_grad(set_to_none=True)
                sch.step()
                step += 1
                if step % 30 == 0:
                    print(
                        f'job {job} ep {ep} step {step}/{total} loss {np.mean(losses[-180:]):.4f} '
                        f'skipped {skipped} ({time.time() - t0:.0f}s)',
                        flush=True,
                    )


### Worker 5 — eval + save (`stage1` / `final` write adapter)


In [ ]:
%%writefile -a worker.py

    model.eval()
    norms.eval()
    Z = np.zeros((len(valid), 4096 * len(L)), np.float32)
    order = np.argsort([len(tokens[i]) for i in valid], kind='stable')
    with torch.inference_mode():
        for s in range(0, len(order), 6):
            pos = order[s:s + 6]
            Z[pos] = rep(valid[pos]).cpu().numpy()
    assert np.isfinite(Z).all()

    W = {n: t.weight.detach().cpu().numpy() for n, t in (('plain', plain), ('center', centered))}
    B = {n: t.bias.detach().cpu().numpy() for n, t in (('plain', plain), ('center', centered))}
    np.savez_compressed(
        os.path.join(outdir, 'readouts.npz'),
        classes=np.array(LABELS),
        pool_layers=np.array(L),
        plain_coef=W['plain'],
        plain_intercept=B['plain'],
        center_coef=W['center'],
        center_intercept=B['center'],
    )
    np.savez_compressed(
        os.path.join(outdir, 'valid_features.npz'),
        ids=train.id.values[valid],
        features=Z.astype(np.float16),
    )
    lg = Z @ W['plain'].T + B['plain']
    acc = float((lg.argmax(1) == y[valid]).mean())

    # save adapter for stage1, final, and (optionally) fold jobs
    if job in ('stage1', 'final') or cfg.get('save_fold_adapters'):
        model.save_pretrained(os.path.join(outdir, 'adapter'), safe_serialization=True)
        save_file(
            {**{f'norms.{k}': v.detach().cpu().contiguous() for k, v in norms.state_dict().items()}},
            os.path.join(outdir, 'norms.safetensors'),
        )
        tok.save_pretrained(os.path.join(outdir, 'tokenizer'))

    json.dump(
        {
            'job': job,
            'valid_acc': acc,
            'steps': step,
            'skipped': skipped,
            'secs': time.time() - t0,
            'init_adapter': cfg.get('init_adapter'),
            'epochs': cfg['epochs'],
            'train_csv': cfg['train_csv'],
        },
        open(os.path.join(outdir, 'COMPLETE.json'), 'w'),
    )
    print(
        f'job {job} done: valid acc {acc:.4f} steps {step} skipped {skipped} ({time.time() - t0:.0f}s)',
        flush=True,
    )
    del model, base, opt, norms, plain, centered
    torch.cuda.empty_cache()


## Launch

1. Find host `train.csv` + Qwen3-8B-Base
2. **Stage 1** on GPU 0 over `pool_train.csv` → `lora_out/job_stage1/` (~4–4.5 h)
3. **Stage 2** on both GPUs (folds + final), warm-started from stage 1 → `lora_out/job_{0..4,final}/`

Use [`Qwen3-8B-base-Lora-submit.ipynb`](Qwen3-8B-base-Lora-submit.ipynb) on `job_final` afterward.

In [ ]:
T0 = time.time()
DATA_DIR = next(
    p
    for p in [
        pathlib.Path('/kaggle/input/competitions/name-that-llm'),
        pathlib.Path('/kaggle/input/name-that-llm'),
        pathlib.Path('data'),
    ]
    if (p / 'train.csv').exists()
)
host_csv = str(DATA_DIR / 'train.csv')
assert pathlib.Path('pool_train.csv').exists(), 'Run the prepare-pool cell first'
assert pathlib.Path('worker.py').exists(), 'Run the %%writefile worker cells first'

model_dir = None
for c in sorted(glob.glob('/kaggle/input/**/config.json', recursive=True)):
    try:
        j = json.load(open(c))
    except Exception:
        continue
    if j.get('model_type') == 'qwen3' and j.get('hidden_size') == 4096 and j.get('num_hidden_layers') == 36:
        model_dir = os.path.dirname(c)
        break
print('base', model_dir, flush=True)
assert model_dir

OUT_ROOT = 'lora_out'


def run_worker(gpu: int, jobs: list, cfg: dict):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
    log = f'worker_gpu{gpu}_{"_".join(jobs)}.log'
    p = subprocess.Popen(
        [sys.executable, 'worker.py', json.dumps(cfg), ','.join(jobs)],
        env=env,
        stdout=open(log, 'w'),
        stderr=subprocess.STDOUT,
    )
    return p, log


# ----- Stage 1: public pool pretrain (single GPU) -----
cfg1 = dict(
    train_csv='pool_train.csv',
    model_dir=model_dir,
    fold_seed=FOLD_SEED,
    seed=SEED,
    layers=LAYERS,
    keep=KEEP,
    max_len=MAX_LEN,
    epochs=S1_EPOCHS,
    accum=ACCUM,
    lr_adapter=S1_LR_ADAPTER,
    lr_head=S1_LR_HEAD,
    wd=WD,
    warmup=WARMUP,
    r=R,
    alpha=ALPHA,
    ldrop=LDROP,
    out_root=OUT_ROOT,
    class_weights=S1_USE_CLASS_WEIGHTS,
    init_adapter=None,
)
print('==== STAGE 1: public pool pretrain ====', flush=True)
p1, log1 = run_worker(0, ['stage1'], cfg1)
rc1 = p1.wait()
print(f'---- stage1 (rc {rc1})')
print(open(log1).read()[-5000:])
assert rc1 == 0, 'stage1 failed'
stage1_adapter = str(pathlib.Path(OUT_ROOT) / 'job_stage1' / 'adapter')
assert pathlib.Path(stage1_adapter).is_dir(), stage1_adapter

# ----- Stage 2: host fine-tune from stage1 -----
cfg2 = dict(
    train_csv=host_csv,
    model_dir=model_dir,
    fold_seed=FOLD_SEED,
    seed=SEED,
    layers=LAYERS,
    keep=KEEP,
    max_len=MAX_LEN,
    epochs=S2_EPOCHS,
    accum=ACCUM,
    lr_adapter=S2_LR_ADAPTER,
    lr_head=S2_LR_HEAD,
    wd=WD,
    warmup=WARMUP,
    r=R,
    alpha=ALPHA,
    ldrop=LDROP,
    out_root=OUT_ROOT,
    class_weights=False,
    init_adapter=stage1_adapter,
)
print('==== STAGE 2: host fine-tune ====', flush=True)
procs = []
for g, jl in S2_JOBS.items():
    procs.append((g, *run_worker(g, jl, cfg2)))
for g, p, log in procs:
    rc = p.wait()
    print(f'---- stage2 worker {g} (rc {rc})')
    print(open(log).read()[-4000:])
    assert rc == 0, f'stage2 gpu {g} failed'

done = sorted(glob.glob(f'{OUT_ROOT}/job_*/COMPLETE.json'))
summary = pd.DataFrame([json.load(open(d)) for d in done])
if not summary.empty:
    cols = [c for c in ['job', 'valid_acc', 'steps', 'skipped', 'secs', 'epochs'] if c in summary.columns]
    summary = summary[cols]
    if 'valid_acc' in summary:
        summary['valid_acc'] = summary['valid_acc'].map(lambda x: f'{x:.4f}')
    if 'secs' in summary:
        summary['secs'] = summary['secs'].map(lambda x: f'{x:.0f}')
print('completed jobs:')
print(summary.to_string(index=False))
print(f'total wall time: {time.time() - T0:.0f}s')
print('Submit with Qwen3-8B-base-Lora-submit.ipynb using lora_out/job_final/')
pd.DataFrame({'id': [], 'model': []}).to_parquet('submission.parquet')